# India Campaign Genome 2026
## Decoding 90 Days of Publicly Reported Indian Marketing Activity

This notebook analyzes a **retrospective, public-source sample** of Indian marketing actions reported between June and August 2026.

The project focuses on **marketing strategy**, not fabricated performance outcomes.

**Research themes:** named-talent dependency, occasion marketing, localization, standardized marketing roles, AI/data-led execution, channel concentration, and creative-strategy diversity.


## 1. Research design

**Unit of analysis:** one publicly reported marketing action.

**Coverage:** actions reported from 1 June to 29 August 2026.

**Sources:** marketing-industry reporting and official brand/platform pages listed in `data/metadata/sources.csv`.

**Manual coding:** sector, objective, channel, creative strategy, original funnel label, and related strategy fields.

**Derived features:** broader `sector_group` categories and standardized `marketing_role` labels are generated reproducibly from the coded data.

> The raw `funnel_stage` field is preserved because it reflects the original coding. It contains a mixture of funnel and strategic labels, so comparative analysis uses the standardized `marketing_role` field instead.

### Research questions

1. Which broad sector groups show the highest named-talent dependency?
2. How important are festivals, awareness days, sports moments, and other occasions in the observed sample?
3. Which actions are explicitly localized by city or region?
4. Where is AI or data materially part of marketing execution?
5. How is observed activity distributed across standardized marketing roles?
6. How concentrated are sector-level channel choices?
7. Which sectors use the widest variety of coded creative strategies?


In [ ]:
from pathlib import Path
import sys
import math

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", None)

# Work whether the notebook is launched from the repository root
# or from inside the notebooks/ directory.
ROOT = Path.cwd()
if not (ROOT / "data").exists():
    ROOT = ROOT.parent

DATA_PATH = ROOT / "data" / "processed" / "campaign_genome_enriched.csv"

df = pd.read_csv(DATA_PATH)
df["reported_date"] = pd.to_datetime(df["reported_date"], errors="coerce")

print("Rows:", len(df))
print("Columns:", df.shape[1])
print("Date range:", df["reported_date"].min().date(), "to", df["reported_date"].max().date())

df.head()


## 2. Data quality and sample snapshot

This dataset is a **public-source reconstruction**, not a census of all Indian advertising.

The following checks help confirm that the processed file is internally consistent before analysis.


In [ ]:
quality_summary = pd.Series({
    "Rows": len(df),
    "Unique record IDs": df["record_id"].nunique(),
    "Duplicate rows": int(df.duplicated().sum()),
    "Missing reported dates": int(df["reported_date"].isna().sum()),
    "Unique brands / partnerships": df["brand"].nunique(),
    "Source-coded sector labels": df["sector"].nunique(),
    "Broad sector groups": df["sector_group"].nunique(),
})

quality_summary


In [ ]:
snapshot = pd.DataFrame({
    "Metric": [
        "Observed marketing actions",
        "Unique brands / partnerships",
        "Source-coded sector labels",
        "Named-talent actions",
        "Occasion-linked actions",
        "AI/data-led actions",
        "Explicitly regional/localized actions",
        "Reported-date range",
    ],
    "Value": [
        f"{len(df)}",
        f"{df['brand'].nunique()}",
        f"{df['sector'].nunique()}",
        f"{int(df['talent_used'].sum())} ({df['talent_used'].mean():.1%})",
        f"{int(df['occasion_flag'].sum())} ({df['occasion_flag'].mean():.1%})",
        f"{int(df['ai_data_angle'].sum())} ({df['ai_data_angle'].mean():.1%})",
        f"{int(df['regional_localization'].sum())} ({df['regional_localization'].mean():.1%})",
        f"{df['reported_date'].min():%d %b %Y} – {df['reported_date'].max():%d %b %Y}",
    ]
})

snapshot


## 3. Where is observed marketing activity concentrated by standardized marketing role?

The original `funnel_stage` field contains a mixture of funnel and strategic labels. For comparison, the repository derives a standardized `marketing_role` variable while preserving the original field.


In [ ]:
role_distribution = df["marketing_role"].value_counts()

display(role_distribution.to_frame("records"))

plt.figure(figsize=(10, 6))
role_distribution.sort_values().plot(kind="barh")
plt.title("Observed Marketing Actions by Standardized Marketing Role")
plt.xlabel("Number of coded actions")
plt.ylabel("")
plt.tight_layout()
plt.show()


## 4. Named-talent dependency

This analysis measures the share of observed actions in each broad sector group that used a named celebrity, athlete, creator, or other public figure.

Only sector groups with **at least 3 observed actions** are shown. Sample sizes are displayed alongside each sector label to make the denominator visible.


In [ ]:
talent = (
    df.groupby("sector_group")["talent_used"]
      .agg(records="size", positives="sum", rate="mean")
      .reset_index()
)

talent = (
    talent[talent["records"] >= 3]
    .sort_values(["rate", "records"], ascending=[False, False])
)

talent["plot_label"] = (
    talent["sector_group"]
    + " (n="
    + talent["records"].astype(str)
    + ")"
)

display(talent[["sector_group", "records", "positives", "rate"]])

talent_series = talent.head(10).set_index("plot_label")["rate"]

plt.figure(figsize=(10, 6))
talent_series.sort_values().plot(kind="barh")
plt.title("Named-Talent Dependency by Broad Sector Group")
plt.xlabel("Share of observed actions using named talent")
plt.ylabel("")
plt.tight_layout()
plt.show()


## 5. Occasion-linked marketing

An action is treated as occasion-linked when the source explicitly connects it to a festival, awareness day, sporting moment, cultural event, or other named occasion.


In [ ]:
occasion_summary = pd.Series({
    "Occasion-linked actions": int(df["occasion_flag"].sum()),
    "Share of observed actions": df["occasion_flag"].mean(),
})

occasion_summary


In [ ]:
occasion_by_month = (
    df.groupby("reported_month")["occasion_flag"]
      .mean()
      .sort_index()
)

display(occasion_by_month.to_frame("occasion_linked_share"))

plt.figure(figsize=(9, 5))
occasion_by_month.plot(kind="bar")
plt.title("Occasion-Linked Share by Reported Month")
plt.xlabel("Reported month")
plt.ylabel("Share of observed actions")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()


## 6. AI & data in marketing execution

The `ai_data_angle` flag is used only when AI or data materially enters the marketing execution, targeting, media, content, or consumer experience—not when AI is merely mentioned as a theme.


In [ ]:
ai_actions = df[df["ai_data_angle"] == 1].copy()

print("AI/data-led actions:", len(ai_actions))
print("Share of observed sample:", f"{len(ai_actions) / len(df):.1%}")

display(
    ai_actions[
        [
            "reported_date",
            "brand",
            "sector_group",
            "campaign_or_action",
            "primary_channel",
            "creative_strategy",
        ]
    ].sort_values("reported_date")
)


In [ ]:
ai_by_sector = (
    ai_actions["sector_group"]
    .value_counts()
)

plt.figure(figsize=(10, 6))
ai_by_sector.sort_values().plot(kind="barh")
plt.title("AI/Data-Led Marketing Actions by Broad Sector Group")
plt.xlabel("Number of coded actions")
plt.ylabel("")
plt.tight_layout()
plt.show()


## 7. Explicit regional localization

This section isolates actions that explicitly target a city, state, region, language market, or regional cultural context.


In [ ]:
regional = df[df["regional_localization"] == 1].copy()

print("Explicitly regional/localized actions:", len(regional))
print("Share of observed sample:", f"{len(regional) / len(df):.1%}")

display(
    regional[
        [
            "reported_date",
            "brand",
            "sector_group",
            "geography_focus",
            "campaign_or_action",
            "creative_strategy",
        ]
    ].sort_values("reported_date")
)


## 8. Channel concentration by sector group

Channel concentration is measured with the **Herfindahl-Hirschman Index (HHI)**:

\[
HHI = \sum_i s_i^2
\]

where \(s_i\) is the share of a sector group's observed actions assigned to channel \(i\).

Higher values indicate a more concentrated observed channel mix.


In [ ]:
def channel_hhi_table(data, group_col="sector_group", channel_col="primary_channel", min_n=3):
    rows = []
    for group, g in data.groupby(group_col):
        if len(g) < min_n:
            continue
        shares = g[channel_col].value_counts(normalize=True)
        rows.append({
            group_col: group,
            "records": len(g),
            "channel_hhi": float((shares ** 2).sum()),
            "distinct_channels": int(g[channel_col].nunique()),
        })
    return pd.DataFrame(rows).sort_values("channel_hhi", ascending=False)

hhi = channel_hhi_table(df)
hhi


## 9. Creative-strategy diversity

Creative-strategy diversity is measured using **Shannon entropy** over the coded `creative_strategy` distribution.

A normalized 0–1 score is also calculated so diversity can be compared more easily across sector groups with different numbers of observed strategy categories.


In [ ]:
def creative_diversity_table(data, group_col="sector_group", strategy_col="creative_strategy", min_n=3):
    rows = []

    for group, g in data.groupby(group_col):
        if len(g) < min_n:
            continue

        shares = g[strategy_col].value_counts(normalize=True)
        entropy = -float(np.sum(shares * np.log(shares)))

        k = len(shares)
        normalized = 0.0 if k <= 1 else entropy / math.log(k)

        rows.append({
            group_col: group,
            "records": len(g),
            "distinct_strategies": k,
            "shannon_entropy": entropy,
            "normalized_diversity": normalized,
        })

    return pd.DataFrame(rows).sort_values(
        ["normalized_diversity", "records"],
        ascending=[False, False],
    )

creative_diversity = creative_diversity_table(df)
creative_diversity


## 10. Key findings

Within this **91-action public-source sample**:

- **Awareness dominates the observed marketing mix:** 43 of 91 actions (47.3%) were classified as awareness-oriented, compared with 23 consideration actions and 11 conversion/acquisition actions.
- **Named talent is common but unevenly distributed:** 36 actions (39.6%) used a named celebrity, athlete, creator, or other public figure. Beauty & Personal Care and Automotive & Mobility showed the highest observed dependency rates, although sector-level sample sizes vary substantially.
- **Occasion marketing is material:** 23 actions (25.3%) were explicitly connected to festivals, awareness days, sports moments, cultural events, or other occasions.
- **Occasion-linked activity increased sharply in August:** roughly 30% of observed August actions were occasion-linked, compared with lower shares in June and July.
- **AI/data-led execution remains concentrated:** 15 actions (16.5%) materially incorporated AI or data into marketing execution, with Technology & Electronics accounting for the largest share of these observed actions.
- **Explicit regional localization was relatively selective:** 12 actions (13.2%) targeted a particular city, state, region, or regional cultural context.

These findings describe the **observed sample only** and should not be interpreted as population estimates for the Indian advertising industry.


## 11. Responsible interpretation and limitations

This repository should **not** be used to claim which campaigns performed best. Outcome metrics such as spend, ROI, impressions, reach, engagement, and sales lift are not consistently public and were not invented.

The dataset is a retrospective reconstruction from public sources, so it may overrepresent campaigns that received marketing-industry coverage.

The `reported_date` field is the reporting/publication date and can differ from the exact campaign launch date.

Several analytical fields—including sector grouping, marketing role, creative strategy, and primary objective—depend on analyst coding. The raw source-level fields are preserved wherever possible so those decisions remain auditable.

Prefer language such as:

> “Within this 91-observation public-source sample…”

rather than broad population claims about all Indian brands or all Indian advertising.
